# Phase 23 — Bilateral (Subject-Level) Graph Model

## Why this notebook exists

Every model built through Phase 22 treats each foot independently and only
combines left/right **after** prediction, by averaging two separate
per-foot probabilities. Phase 3's original EDA found that left-right
temperature **asymmetry** is one of the strongest DM-associated signals in
the dataset (mean 0.711°C in DM vs 0.316°C in Control, Mann-Whitney
p=0.0031) and explicitly flagged it as an unused candidate graph-edge
feature. This notebook is the first place that signal is actually given to
the model.

Instead of two independent 4-node per-foot graphs, this notebook builds
**one 8-node graph per SUBJECT**: 4 nodes for the left foot's angiosome
regions, 4 for the right foot's, connected by the existing within-foot
anatomical adjacency **plus** 4 new cross-foot "mirror" edges (each region
to its counterpart on the other foot) carrying the asymmetry signal
directly as an edge feature. Because each sample now covers a whole
subject, predictions are natively subject-level — no post-hoc foot
averaging is needed.

**Node indexing convention (used exactly, everywhere in this notebook):**

| index | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|---|
| node | LCA_L | LPA_L | MPA_L | MCA_L | LCA_R | LPA_R | MPA_R | MCA_R |

This notebook reads (never writes to):
- `data/processed/subject_metadata.csv`
- `data/processed/graphs/{subject_id}_L.npz` / `_R.npz`
- `data/processed/graph_manifest.csv`
- `data/processed/normalization_stats.json`

It imports `GraphBranch` and `sparse_to_dense_graph` from `src/model.py`
directly, but **not** `DualBranchGAT` or `build_model`/`train_model`/
`evaluate` from `src/train.py` — this model's forward signature (8 nodes,
no image branch, no `graph_topology`/`image_key` dispatch) is genuinely
different, so it gets its own small training loop rather than forcing a
reuse that doesn't fit.


## Section 1 — Setup

Standard project conventions: fixed random seeds, device detection, and
loading the three already-validated source-of-truth files from prior
phases (subject metadata, the per-foot graph manifest, and the train-split
normalization statistics from Phase 7 — reused as-is, never recomputed
here).


In [1]:
print("=== SECTION 1: Setup ===")

import sys
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score, accuracy_score

import matplotlib.pyplot as plt

# --- Reproducibility ---
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# --- Project imports ---
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.model import GraphBranch, sparse_to_dense_graph

# --- Paths (READ-ONLY -- this notebook never writes to data/processed/) ---
DATA_DIR = Path("data/processed")
METADATA_PATH = DATA_DIR / "subject_metadata.csv"
GRAPH_DIR = DATA_DIR / "graphs"
MANIFEST_PATH = DATA_DIR / "graph_manifest.csv"
NORM_STATS_PATH = DATA_DIR / "normalization_stats.json"

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- Load source-of-truth files ---
subject_metadata_df = pd.read_csv(METADATA_PATH)
graph_manifest_df = pd.read_csv(MANIFEST_PATH)
with open(NORM_STATS_PATH, "r") as f:
    normalization_stats = json.load(f)

print(f"subject_metadata_df shape: {subject_metadata_df.shape}")
print(f"graph_manifest_df shape: {graph_manifest_df.shape}")
print(f"normalization_stats keys: {list(normalization_stats.keys())}")

# Subject -> {0,1} label lookup (1 = DM, 0 = Control), reused throughout
subject_label_map = dict(
    zip(subject_metadata_df["Subject"], (subject_metadata_df["Group"] == "DM").astype(int))
)
print(f"subject_label_map built for {len(subject_label_map)} subjects")


=== SECTION 1: Setup ===
Device: cuda
subject_metadata_df shape: (167, 19)
graph_manifest_df shape: (334, 9)
normalization_stats keys: ['node_feature_names', 'node_feature_mean', 'node_feature_std', 'raw_tissue_mean', 'raw_tissue_std', 'relative_tissue_mean', 'relative_tissue_std', 'computed_from_split', 'train_subject_foot_count']
subject_label_map built for 167 subjects


## Section 2 — Build the 8-Node Edge Structure

Fixed and identical for every subject: an 8-cycle within each foot (mirror
images of each other, right shifted by +4) plus 4 undirected cross-foot
"mirror" edges connecting each region to its counterpart on the other
foot. Every undirected edge is stored as two directed entries, giving
`edge_index` shape `(2, 24)`: 8 left-cycle + 8 right-cycle + 8 cross-foot
directed edges.

Printed in full below so the topology can be visually double-checked
against the node-indexing convention stated above.


In [2]:
print("=== SECTION 2: 8-Node Edge Structure ===")

NODE_NAMES = ["LCA_L", "LPA_L", "MPA_L", "MCA_L", "LCA_R", "LPA_R", "MPA_R", "MCA_R"]
# 0=LCA_L, 1=LPA_L, 2=MPA_L, 3=MCA_L, 4=LCA_R, 5=LPA_R, 6=MPA_R, 7=MCA_R

within_left = [(0, 1), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2), (3, 0), (0, 3)]
within_right = [(4, 5), (5, 4), (5, 6), (6, 5), (6, 7), (7, 6), (7, 4), (4, 7)]
cross_mirror = [(0, 4), (4, 0), (1, 5), (5, 1), (2, 6), (6, 2), (3, 7), (7, 3)]

all_edges = within_left + within_right + cross_mirror
edge_index = np.array(all_edges, dtype=np.int64).T  # shape (2, 24)

NUM_WITHIN_EDGES = len(within_left) + len(within_right)  # 16
NUM_CROSS_EDGES = len(cross_mirror)  # 8

assert edge_index.shape == (2, 24), f"Expected (2,24), got {edge_index.shape}"

print("Node order:")
for i, n in enumerate(NODE_NAMES):
    print(f"  {i} = {n}")

print(f"\nedge_index shape: {edge_index.shape}")
print("Full edge_index (row 0 = source, row 1 = target):")
print(edge_index)

print(f"\nWithin-foot directed edges (indices 0-{NUM_WITHIN_EDGES - 1}): {NUM_WITHIN_EDGES}")
print(f"Cross-foot mirror directed edges (indices {NUM_WITHIN_EDGES}-23): {NUM_CROSS_EDGES}")


=== SECTION 2: 8-Node Edge Structure ===
Node order:
  0 = LCA_L
  1 = LPA_L
  2 = MPA_L
  3 = MCA_L
  4 = LCA_R
  5 = LPA_R
  6 = MPA_R
  7 = MCA_R

edge_index shape: (2, 24)
Full edge_index (row 0 = source, row 1 = target):
[[0 1 1 2 2 3 3 0 4 5 5 6 6 7 7 4 0 4 1 5 2 6 3 7]
 [1 0 2 1 3 2 0 3 5 4 6 5 7 6 4 7 4 0 5 1 6 2 7 3]]

Within-foot directed edges (indices 0-15): 16
Cross-foot mirror directed edges (indices 16-23): 8


## Section 3 — Per-Subject Node Features and Edge Attributes

For every subject: load both feet's raw `(4,6)` node features from their
`.npz` files, concatenate into `(8,6)` in the fixed node order above, and
z-score normalize using the single train-derived 6-value mean/std from
`normalization_stats.json` (applied identically to all 8 rows, since the
per-region feature semantics are the same regardless of foot side).

Two different edge-feature formulas are used, matching the two edge
*kinds*:

- **Within-foot edges** (the 16 left-cycle + right-cycle directed edges):
  reuse the exact Phase 6 formula — `[0]` = Euclidean distance between the
  two nodes' `(centroid_row_norm, centroid_col_norm)` (raw, unnormalized
  columns 4,5), `[1]` = absolute difference of `mean_abs_temp` (raw
  column 0).
- **Cross-foot mirror edges** (the 8 L↔R directed edges): a different
  formula that directly operationalizes the asymmetry signal — `[0]` =
  absolute difference of `mean_abs_temp` (raw column 0) between the L node
  and its R counterpart, `[1]` = absolute difference of `mean_rel_temp`
  (raw column 2) between the L node and its R counterpart.

**Design note:** `edge_index` is fixed and identical for every subject
(built once in Section 2), so `features_dict` below stores only
`(node_features, edge_attr)` per subject rather than re-storing the same
24-edge index 167 times — every downstream cell reuses the module-level
`edge_index` directly.


In [3]:
print("=== SECTION 3: Per-Subject Node Features and Edge Attributes ===")

# subject_id -> {"L": npz_path, "R": npz_path}
manifest_lookup = {}
for _, row in graph_manifest_df.iterrows():
    sid = row["subject_id"]
    side = row["foot_side"]
    manifest_lookup.setdefault(sid, {})[side] = row["npz_path"]

node_feature_mean = np.array(normalization_stats["node_feature_mean"], dtype=np.float64)
node_feature_std = np.array(normalization_stats["node_feature_std"], dtype=np.float64)


def build_subject_graph(subject_id: str):
    """
    Load raw (4,6) node_features for both feet, concatenate into (8,6) in
    the fixed node order, z-score normalize, and compute edge_attr (24,2)
    for the fixed 8-node edge structure from Section 2.
    """
    paths = manifest_lookup[subject_id]
    l_npz = np.load(paths["L"])
    r_npz = np.load(paths["R"])

    l_raw = l_npz["node_features"]  # (4,6) raw
    r_raw = r_npz["node_features"]  # (4,6) raw

    raw_node_features = np.concatenate([l_raw, r_raw], axis=0)  # (8,6)
    norm_node_features = (raw_node_features - node_feature_mean) / node_feature_std

    edge_attr = np.zeros((24, 2), dtype=np.float64)

    # Within-foot edges (indices 0..NUM_WITHIN_EDGES-1): existing Phase 6 formula
    for e_idx, (i, j) in enumerate(all_edges[:NUM_WITHIN_EDGES]):
        ci = raw_node_features[i, 4:6]
        cj = raw_node_features[j, 4:6]
        dist = np.linalg.norm(ci - cj)
        temp_grad = abs(raw_node_features[i, 0] - raw_node_features[j, 0])
        edge_attr[e_idx, 0] = dist
        edge_attr[e_idx, 1] = temp_grad

    # Cross-foot mirror edges (last NUM_CROSS_EDGES): asymmetry formula
    for offset, (i, j) in enumerate(all_edges[NUM_WITHIN_EDGES:]):
        e_idx = NUM_WITHIN_EDGES + offset
        abs_temp_diff = abs(raw_node_features[i, 0] - raw_node_features[j, 0])
        rel_temp_diff = abs(raw_node_features[i, 2] - raw_node_features[j, 2])
        edge_attr[e_idx, 0] = abs_temp_diff
        edge_attr[e_idx, 1] = rel_temp_diff

    return norm_node_features.astype(np.float32), edge_attr.astype(np.float32)


# Apply to all 167 subjects
subject_ids = subject_metadata_df["Subject"].values
features_dict = {}
n_pass, n_fail = 0, 0
fail_list = []

for sid in subject_ids:
    try:
        nf, ea = build_subject_graph(sid)
        ok_shape = (nf.shape == (8, 6)) and (ea.shape == (24, 2))
        ok_nan = (not np.isnan(nf).any()) and (not np.isnan(ea).any())
        if ok_shape and ok_nan:
            features_dict[sid] = (nf, ea)
            n_pass += 1
        else:
            n_fail += 1
            fail_list.append(sid)
    except Exception as e:
        n_fail += 1
        fail_list.append(sid)
        print(f"  ERROR building graph for {sid}: {e}")

print(f"PASS: {n_pass} / {len(subject_ids)} subjects (shape (8,6)/(24,2), no NaNs)")
print(f"FAIL: {n_fail} / {len(subject_ids)} subjects")
if fail_list:
    print("Failed subjects:", fail_list)


=== SECTION 3: Per-Subject Node Features and Edge Attributes ===
PASS: 167 / 167 subjects (shape (8,6)/(24,2), no NaNs)
FAIL: 0 / 167 subjects


## Section 4 — Mandatory Smoke Test (READ CAREFULLY, DO NOT SKIP)

`GraphBranch` and `sparse_to_dense_graph` were built and validated in
Phase 8 against 4-node per-foot graphs. Nothing guarantees they generalize
cleanly to an 8-node graph without being told so explicitly — this cell is
the single point where that assumption is actually tested, on a small
4-subject batch, **before** anything downstream is built on top of it.

If the imported `sparse_to_dense_graph` utility raises an error or
produces an unexpected shape, the code below does **not** attempt to patch
`src/model.py`. Instead it falls back to building the dense conversion
manually, entirely within this notebook, exactly per the specified logic:
an all-`False` `(B,8,8)` mask with self-loops set `True` on the diagonal
and each real edge set `True` at its `(i,j)` position, and an all-zero
`(B,8,8,2)` edge-attribute tensor with each real edge's `[d, g]` written to
its `(i,j)` position (self-loops and non-edges correctly stay `[0,0]`).


In [4]:
print("=== SECTION 4: Mandatory Smoke Test ===")

SMOKE_SUBJECTS = list(features_dict.keys())[:4]

smoke_node_features = torch.tensor(
    np.stack([features_dict[s][0] for s in SMOKE_SUBJECTS]), dtype=torch.float32
).to(device)  # (4,8,6)

smoke_edge_index_np = np.tile(edge_index, (len(SMOKE_SUBJECTS), 1, 1))  # (4,2,24)
smoke_edge_index = torch.tensor(smoke_edge_index_np, dtype=torch.int64).to(device)

smoke_edge_attr = torch.tensor(
    np.stack([features_dict[s][1] for s in SMOKE_SUBJECTS]), dtype=torch.float32
).to(device)  # (4,24,2)

print(f"smoke_node_features shape: {tuple(smoke_node_features.shape)}")
print(f"smoke_edge_index shape:    {tuple(smoke_edge_index.shape)}")
print(f"smoke_edge_attr shape:     {tuple(smoke_edge_attr.shape)}")

extraction_path_used = None
extraction_reason = None

try:
    dense_mask, dense_edge_attr = sparse_to_dense_graph(smoke_edge_index, smoke_edge_attr, num_nodes=8)
    graph_branch_smoke = GraphBranch(node_in_dim=6, hidden_dim=32, dropout_p=0.3).to(device)
    smoke_output = graph_branch_smoke(smoke_node_features, dense_mask, dense_edge_attr)

    print(f"\nOutput shape: {tuple(smoke_output.shape)} (expected (4, 32))")
    assert smoke_output.shape == (4, 32), f"Unexpected output shape: {smoke_output.shape}"
    assert not torch.isnan(smoke_output).any(), "NaNs in smoke test output"

    extraction_path_used = "imported utility (src.model.sparse_to_dense_graph)"
    extraction_reason = (
        "sparse_to_dense_graph(edge_index, edge_attr, num_nodes=8) ran without error "
        "and produced the expected (4, 32) shape with no NaNs -- no fallback needed."
    )
    print("PASS: imported utility path succeeded, correct shape, no NaNs.")

except Exception as e:
    print(f"\nImported utility path FAILED: {type(e).__name__}: {e}")
    print("Falling back to manual dense-graph construction (built entirely in this notebook)...")

    B, N = len(SMOKE_SUBJECTS), 8
    dense_mask = torch.zeros((B, N, N), dtype=torch.bool)
    dense_edge_attr = torch.zeros((B, N, N, 2), dtype=torch.float32)

    edge_index_np = smoke_edge_index.detach().cpu().numpy()
    edge_attr_np = smoke_edge_attr.detach().cpu().numpy()

    for b in range(B):
        for i in range(N):
            dense_mask[b, i, i] = True  # self-loops; edge_attr stays [0,0] by construction
        for e in range(edge_index_np.shape[2]):
            i, j = int(edge_index_np[b, 0, e]), int(edge_index_np[b, 1, e])
            dense_mask[b, i, j] = True
            dense_edge_attr[b, i, j, :] = torch.tensor(edge_attr_np[b, e, :])

    dense_mask = dense_mask.to(device)
    dense_edge_attr = dense_edge_attr.to(device)

    graph_branch_smoke = GraphBranch(node_in_dim=6, hidden_dim=32, dropout_p=0.3).to(device)
    smoke_output = graph_branch_smoke(smoke_node_features, dense_mask, dense_edge_attr)

    print(f"\nOutput shape (manual fallback): {tuple(smoke_output.shape)} (expected (4, 32))")
    assert smoke_output.shape == (4, 32), f"Unexpected output shape: {smoke_output.shape}"
    assert not torch.isnan(smoke_output).any(), "NaNs in smoke test output"

    extraction_path_used = "manual fallback (built entirely within this notebook)"
    extraction_reason = (
        "the imported sparse_to_dense_graph utility raised an error or produced an "
        "unexpected shape for num_nodes=8, so dense conversion was built manually per "
        "the notebook's specified fallback logic instead of patching src/model.py."
    )
    print("PASS: manual fallback path succeeded, correct shape, no NaNs.")

print(f"\nExtraction path used: {extraction_path_used}")
print(f"Reason: {extraction_reason}")


=== SECTION 4: Mandatory Smoke Test ===
smoke_node_features shape: (4, 8, 6)
smoke_edge_index shape:    (4, 2, 24)
smoke_edge_attr shape:     (4, 24, 2)

Output shape: (4, 32) (expected (4, 32))
PASS: imported utility path succeeded, correct shape, no NaNs.

Extraction path used: imported utility (src.model.sparse_to_dense_graph)
Reason: sparse_to_dense_graph(edge_index, edge_attr, num_nodes=8) ran without error and produced the expected (4, 32) shape with no NaNs -- no fallback needed.


**Which extraction path was actually used (fill in after running the cell above):**

- Path used: *`imported utility` or `manual fallback` — copy the exact
  string printed above.*
- Reason: *copy the printed reason above.*

This determines which branch of `get_dense_graph` (defined in Section 6)
is exercised for every training/evaluation batch from Section 8 onward —
the code itself checks `extraction_path_used` at call time, so no manual
code change is needed either way, but this is the human-readable record of
which path this run actually took.


## Section 5 — BilateralGraphClassifier Model

A small model defined in this notebook (not `src/model.py`): the same
`GraphBranch(node_in_dim=6, hidden_dim=32, dropout_p=0.3)` used everywhere
else in this project, followed by the same classifier head shape as
Phase 8's fusion model when only the graph branch is active
(`Linear(32,16) -> ReLU -> Dropout(0.3) -> Linear(16,1)`), squeezed to
shape `(B,)`.


In [5]:
print("=== SECTION 5: BilateralGraphClassifier ===")


class BilateralGraphClassifier(nn.Module):
    def __init__(self, node_in_dim=6, hidden_dim=32, dropout_p=0.3):
        super().__init__()
        self.graph_branch = GraphBranch(node_in_dim=node_in_dim, hidden_dim=hidden_dim, dropout_p=dropout_p)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 16),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(16, 1),
        )

    def forward(self, node_features, dense_mask, dense_edge_attr):
        graph_embedding = self.graph_branch(node_features, dense_mask, dense_edge_attr)
        logits = self.classifier(graph_embedding)
        return logits.squeeze(-1)


bilateral_model_probe = BilateralGraphClassifier(node_in_dim=6, hidden_dim=32, dropout_p=0.3)
total_params = sum(p.numel() for p in bilateral_model_probe.parameters())
print(f"BilateralGraphClassifier total parameter count: {total_params}")

PHASE12_GRAPH_ONLY_BASE_PARAMS = 6465
if total_params == PHASE12_GRAPH_ONLY_BASE_PARAMS:
    print(f"PASS: matches Phase 12's graph_only_base count ({PHASE12_GRAPH_ONLY_BASE_PARAMS}) exactly.")
else:
    print(f"NOTE: does NOT match Phase 12's graph_only_base count ({PHASE12_GRAPH_ONLY_BASE_PARAMS}) -- see markdown note below.")

del bilateral_model_probe


=== SECTION 5: BilateralGraphClassifier ===
BilateralGraphClassifier total parameter count: 6465
PASS: matches Phase 12's graph_only_base count (6465) exactly.


**Parameter count note:** `BilateralGraphClassifier`'s graph branch is
architecturally identical to Phase 8's `GraphBranch(node_in_dim=6,
hidden_dim=32, dropout_p=0.3)`, just applied to 8 nodes instead of 4, and
the classifier head (`Linear(32,16) -> ReLU -> Dropout -> Linear(16,1)`)
is the same shape Phase 12 used for its single-branch (`graph_only_base`)
ablation config. Every learned weight in an `EdgeGATLayer`
(`W_node`, `W_edge`, the attention vector `a`) is shared across nodes —
the same projection is applied to every node's features independently —
so the total parameter count does **not** depend on how many nodes are in
the graph; node count only changes the shape of the computation graph at
forward time, not how many weights exist. The printed total above should
therefore equal Phase 12's `graph_only_base` count of 6,465 exactly, which
the cell above checks directly rather than only asserting it silently.


## Section 6 — Train/Eval Loop Functions

Written fresh for this notebook rather than reused from `src/train.py`,
since that module's `train_one_epoch`/`evaluate` expect `DualBranchGAT`'s
different forward signature (`graph_topology`/`image_key` dispatch,
optional image branch) that doesn't apply here.

`get_dense_graph` reuses whichever extraction path Section 4's smoke test
validated (checked at call time via `extraction_path_used`), so every
batch in every fold goes through the same, already-tested conversion
logic. Because this notebook has no ready-made `sklearn.metrics.roc_curve`
in its allowed import list, the Youden's-J-optimal threshold used in
Section 9 is computed directly with a small self-contained function
(`youdens_j_threshold`) rather than importing it.


In [6]:
print("=== SECTION 6: Train/Eval Loop Functions ===")


def get_dense_graph(node_features, edge_index_batch, edge_attr_batch, device):
    """Dense-graph conversion, reusing whichever path Section 4 validated."""
    if extraction_path_used is not None and extraction_path_used.startswith("imported"):
        return sparse_to_dense_graph(edge_index_batch, edge_attr_batch, num_nodes=8)

    B, N = node_features.shape[0], 8
    edge_index_np = edge_index_batch.detach().cpu().numpy()
    edge_attr_np = edge_attr_batch.detach().cpu().numpy()

    dense_mask = torch.zeros((B, N, N), dtype=torch.bool)
    dense_edge_attr = torch.zeros((B, N, N, 2), dtype=torch.float32)

    for b in range(B):
        for i in range(N):
            dense_mask[b, i, i] = True
        for e in range(edge_index_np.shape[2]):
            i, j = int(edge_index_np[b, 0, e]), int(edge_index_np[b, 1, e])
            dense_mask[b, i, j] = True
            dense_edge_attr[b, i, j, :] = torch.tensor(edge_attr_np[b, e, :])

    return dense_mask.to(device), dense_edge_attr.to(device)


def batch_subjects(subject_ids, features_dict, batch_size=16):
    """Simple list-batching (~16 subjects at a time) over the fixed edge_index."""
    subject_ids = list(subject_ids)
    for start in range(0, len(subject_ids), batch_size):
        batch_ids = subject_ids[start:start + batch_size]
        nf = torch.tensor(np.stack([features_dict[s][0] for s in batch_ids]), dtype=torch.float32)
        ea = torch.tensor(np.stack([features_dict[s][1] for s in batch_ids]), dtype=torch.float32)
        ei = torch.tensor(np.tile(edge_index, (len(batch_ids), 1, 1)), dtype=torch.int64)
        yield batch_ids, nf, ei, ea


def train_one_epoch_bilateral(model, subject_ids, features_dict, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    n_batches = 0

    for batch_ids, nf, ei, ea in batch_subjects(subject_ids, features_dict, batch_size=16):
        nf, ei, ea = nf.to(device), ei.to(device), ea.to(device)
        labels = torch.tensor([float(subject_label_map[s]) for s in batch_ids], dtype=torch.float32).to(device)

        dense_mask, dense_edge_attr = get_dense_graph(nf, ei, ea, device)

        optimizer.zero_grad()
        logits = model(nf, dense_mask, dense_edge_attr)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


def evaluate_bilateral(model, subject_ids, features_dict, criterion, device):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    all_labels, all_probs, all_ids = [], [], []

    with torch.no_grad():
        for batch_ids, nf, ei, ea in batch_subjects(subject_ids, features_dict, batch_size=16):
            nf, ei, ea = nf.to(device), ei.to(device), ea.to(device)
            labels = torch.tensor([float(subject_label_map[s]) for s in batch_ids], dtype=torch.float32).to(device)

            dense_mask, dense_edge_attr = get_dense_graph(nf, ei, ea, device)
            logits = model(nf, dense_mask, dense_edge_attr)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            n_batches += 1

            probs = torch.sigmoid(logits).detach().cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
            all_ids.extend(batch_ids)

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    preds = (all_probs >= 0.5).astype(int)

    metrics = {
        "loss": total_loss / max(n_batches, 1),
        "accuracy": accuracy_score(all_labels, preds),
        "balanced_accuracy": balanced_accuracy_score(all_labels, preds),
        "roc_auc": roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else float("nan"),
        "f1": f1_score(all_labels, preds),
        "labels": all_labels,
        "probs": all_probs,
        "subject_ids": all_ids,
    }
    return metrics


def youdens_j_threshold(labels, probs):
    """Small self-contained Youden's-J-optimal threshold search (no sklearn.metrics.roc_curve)."""
    labels = np.asarray(labels)
    probs = np.asarray(probs)
    thresholds = np.unique(probs)

    best_threshold = 0.5
    best_j = -np.inf
    for t in thresholds:
        preds = (probs >= t).astype(int)
        tp = np.sum((preds == 1) & (labels == 1))
        fn = np.sum((preds == 0) & (labels == 1))
        fp = np.sum((preds == 1) & (labels == 0))
        tn = np.sum((preds == 0) & (labels == 0))
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        j = tpr - fpr
        if j > best_j:
            best_j = j
            best_threshold = t

    return float(best_threshold), float(best_j)


print("Defined: get_dense_graph, batch_subjects, train_one_epoch_bilateral, evaluate_bilateral, youdens_j_threshold")


=== SECTION 6: Train/Eval Loop Functions ===
Defined: get_dense_graph, batch_subjects, train_one_epoch_bilateral, evaluate_bilateral, youdens_j_threshold


## Section 7 — Reproduce the Same 5 Folds as Phase 13/17

Identical fold-construction method to every prior CV notebook in this
project: `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` on
`subject_metadata.csv`'s `Subject`/`Group` columns in original row order,
each fold's train portion further split 90/10 (stratified,
`random_state=42`) into train/val.


In [7]:
print("=== SECTION 7: Reproduce 5-Fold Splits ===")

subject_ids_all = subject_metadata_df["Subject"].values
groups_all = subject_metadata_df["Group"].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
folds = []

for fold_idx, (trainval_idx, test_idx) in enumerate(skf.split(subject_ids_all, groups_all)):
    trainval_ids = subject_ids_all[trainval_idx]
    trainval_groups = groups_all[trainval_idx]
    test_ids = subject_ids_all[test_idx]

    train_ids, val_ids = train_test_split(
        trainval_ids,
        test_size=0.10,
        stratify=trainval_groups,
        random_state=RANDOM_SEED,
    )

    folds.append({"train": train_ids, "val": val_ids, "test": test_ids})
    print(f"Fold {fold_idx}: train={len(train_ids)}, val={len(val_ids)}, test={len(test_ids)}")

# Leakage check: every subject appears in exactly one fold's test set
all_test_ids = np.concatenate([f["test"] for f in folds])
leakage_pass = (len(all_test_ids) == len(subject_ids_all)) and (len(set(all_test_ids)) == len(subject_ids_all))

print(f"\nTotal subjects covered across all fold test sets: {len(all_test_ids)} (expected {len(subject_ids_all)})")
print(f"Unique subjects covered: {len(set(all_test_ids))} (expected {len(subject_ids_all)})")
print(f"LEAKAGE CHECK: {'PASS' if leakage_pass else 'FAIL'}")
assert leakage_pass, "Fold leakage check failed -- stop and investigate before training."


=== SECTION 7: Reproduce 5-Fold Splits ===
Fold 0: train=119, val=14, test=34
Fold 1: train=119, val=14, test=34
Fold 2: train=120, val=14, test=33
Fold 3: train=120, val=14, test=33
Fold 4: train=120, val=14, test=33

Total subjects covered across all fold test sets: 167 (expected 167)
Unique subjects covered: 167 (expected 167)
LEAKAGE CHECK: PASS


## Section 8 — Train 5 Folds

For each fold: recompute class weights from that fold's train subjects
only (the same inverse-frequency formula as every prior phase,
`weight_c = N_train / (2 * count_c)`), use `pos_weight = weight_DM`
(DM is the majority class here too — same downweight-the-majority
reasoning established in Phase 10), train with Adam
(`lr=1e-3, weight_decay=1e-4`) for up to 150 epochs with early stopping
on val ROC-AUC (`patience=20`), and save the best checkpoint to
`checkpoints/bilateral_fold{idx}_best.pt`.


In [8]:
print("=== SECTION 8: Train 5 Folds ===")


def train_model_bilateral(train_ids, val_ids, criterion, device,
                           max_epochs=150, patience=20, lr=1e-3, weight_decay=1e-4,
                           checkpoint_path=None):
    model = BilateralGraphClassifier(node_in_dim=6, hidden_dim=32, dropout_p=0.3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_roc_auc = -np.inf
    best_epoch = -1
    epochs_since_improvement = 0
    history = []

    for epoch in range(max_epochs):
        train_loss = train_one_epoch_bilateral(model, train_ids, features_dict, optimizer, criterion, device)
        val_metrics = evaluate_bilateral(model, val_ids, features_dict, criterion, device)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_roc_auc": val_metrics["roc_auc"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        })

        improved = val_metrics["roc_auc"] > best_val_roc_auc
        if improved:
            best_val_roc_auc = val_metrics["roc_auc"]
            best_epoch = epoch
            epochs_since_improvement = 0
            if checkpoint_path is not None:
                torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_since_improvement += 1

        if epochs_since_improvement >= patience:
            print(f"  Early stopping at epoch {epoch} (best epoch {best_epoch}, best val ROC-AUC {best_val_roc_auc:.4f})")
            break

    if epochs_since_improvement < patience:
        print(f"  Reached max_epochs={max_epochs} (best epoch {best_epoch}, best val ROC-AUC {best_val_roc_auc:.4f})")

    return history, best_epoch, best_val_roc_auc


fold_results = []

for fold_idx, fold in enumerate(folds):
    print(f"\n--- Fold {fold_idx + 1}/5 ---")
    train_ids = fold["train"]

    train_groups = subject_metadata_df.set_index("Subject").loc[train_ids, "Group"]
    n_train = len(train_groups)
    n_dm = int((train_groups == "DM").sum())
    n_control = int((train_groups == "Control").sum())
    weight_dm = n_train / (2 * n_dm)
    weight_control = n_train / (2 * n_control)
    print(f"  Class weights (train n={n_train}, DM={n_dm}, Control={n_control}): "
          f"Control={weight_control:.4f}, DM={weight_dm:.4f} -- pos_weight=DM's own weight (downweights majority)")

    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weight_dm, dtype=torch.float32).to(device))

    checkpoint_path = CHECKPOINT_DIR / f"bilateral_fold{fold_idx}_best.pt"
    history, best_epoch, best_val_roc_auc = train_model_bilateral(
        train_ids, fold["val"], criterion, device,
        max_epochs=150, patience=20, lr=1e-3, weight_decay=1e-4,
        checkpoint_path=checkpoint_path,
    )

    fold_results.append({
        "fold": fold_idx,
        "best_epoch": best_epoch,
        "best_val_roc_auc": best_val_roc_auc,
        "history": history,
        "checkpoint_path": str(checkpoint_path),
        "weight_control": weight_control,
        "weight_dm": weight_dm,
    })
    print(f"  Fold {fold_idx} FINAL: best epoch={best_epoch}, best val ROC-AUC={best_val_roc_auc:.4f}")

print("\nAll 5 folds trained.")


=== SECTION 8: Train 5 Folds ===

--- Fold 1/5 ---
  Class weights (train n=119, DM=87, Control=32): Control=1.8594, DM=0.6839 -- pos_weight=DM's own weight (downweights majority)
  Early stopping at epoch 38 (best epoch 18, best val ROC-AUC 0.8250)
  Fold 0 FINAL: best epoch=18, best val ROC-AUC=0.8250

--- Fold 2/5 ---
  Class weights (train n=119, DM=87, Control=32): Control=1.8594, DM=0.6839 -- pos_weight=DM's own weight (downweights majority)
  Early stopping at epoch 23 (best epoch 3, best val ROC-AUC 1.0000)
  Fold 1 FINAL: best epoch=3, best val ROC-AUC=1.0000

--- Fold 3/5 ---
  Class weights (train n=120, DM=88, Control=32): Control=1.8750, DM=0.6818 -- pos_weight=DM's own weight (downweights majority)
  Early stopping at epoch 20 (best epoch 0, best val ROC-AUC 0.9750)
  Fold 2 FINAL: best epoch=0, best val ROC-AUC=0.9750

--- Fold 4/5 ---
  Class weights (train n=120, DM=88, Control=32): Control=1.8750, DM=0.6818 -- pos_weight=DM's own weight (downweights majority)
  Early 

## Section 9 — Test Evaluation

For each fold: reload the best checkpoint, compute that fold's own
Youden's-J-optimal threshold from its own val predictions, and evaluate on
that fold's test subjects with a **plain deterministic forward pass**.

**Scoping decision, not an oversight:** MC-Dropout is skipped for this
first pass. Every other model in this project uses `mc_dropout_predict`
for its final reported test numbers, but this notebook's job is to
establish whether the bilateral 8-node representation is worth pursuing at
all — a deterministic pass is sufficient to answer that, and MC-Dropout
uncertainty quantification for the bilateral model (if it proves worth
keeping) is left to a later phase rather than bundled in here.

Predictions are naturally subject-level (one graph per subject, no
per-foot averaging), so the pooled 167-subject array is built directly
from each fold's test predictions with no extra aggregation step.


In [9]:
print("=== SECTION 9: Test Evaluation ===")

pooled_records = []
subject_group_lookup = subject_metadata_df.set_index("Subject")["Group"]

for fold_idx, fold in enumerate(folds):
    print(f"\n--- Fold {fold_idx} test evaluation ---")

    model = BilateralGraphClassifier(node_in_dim=6, hidden_dim=32, dropout_p=0.3).to(device)
    checkpoint_path = fold_results[fold_idx]["checkpoint_path"]
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    weight_dm = fold_results[fold_idx]["weight_dm"]
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weight_dm, dtype=torch.float32).to(device))

    val_metrics = evaluate_bilateral(model, fold["val"], features_dict, criterion, device)
    optimal_threshold, optimal_j = youdens_j_threshold(val_metrics["labels"], val_metrics["probs"])
    print(f"  Optimal threshold (Youden's J={optimal_j:.4f}, from val): {optimal_threshold:.4f}")

    test_metrics = evaluate_bilateral(model, fold["test"], features_dict, criterion, device)
    test_preds_optimal = (test_metrics["probs"] >= optimal_threshold).astype(int)

    fold_roc_auc = test_metrics["roc_auc"]
    fold_bal_acc = balanced_accuracy_score(test_metrics["labels"], test_preds_optimal)
    fold_f1 = f1_score(test_metrics["labels"], test_preds_optimal)
    fold_acc = accuracy_score(test_metrics["labels"], test_preds_optimal)

    print(f"  Test (n={len(test_metrics['subject_ids'])}): roc_auc={fold_roc_auc:.4f}, "
          f"balanced_accuracy={fold_bal_acc:.4f}, f1={fold_f1:.4f}, accuracy={fold_acc:.4f}")

    for sid, label, prob, pred in zip(test_metrics["subject_ids"], test_metrics["labels"],
                                       test_metrics["probs"], test_preds_optimal):
        pooled_records.append({
            "subject_id": sid,
            "group": subject_group_lookup.loc[sid],
            "true_label": int(label),
            "bilateral_prob": float(prob),
            "pred_label_optimal": int(pred),
            "fold": fold_idx,
            "fold_roc_auc": fold_roc_auc,
            "fold_balanced_accuracy": fold_bal_acc,
            "fold_f1": fold_f1,
            "fold_accuracy": fold_acc,
            "fold_optimal_threshold": float(optimal_threshold),
        })

pooled_df = pd.DataFrame(pooled_records)

print(f"\nPooled predictions: {len(pooled_df)} subjects (expected 167)")
assert len(pooled_df) == 167, f"Expected 167 pooled subjects, got {len(pooled_df)}"
assert pooled_df["subject_id"].nunique() == 167, "Duplicate subject_id in pooled predictions"
print("PASS: pooled predictions cover exactly 167 unique subjects.")

per_fold_summary = pooled_df.groupby("fold")[["fold_roc_auc", "fold_balanced_accuracy", "fold_f1", "fold_accuracy"]].first()

fold_mean_roc_auc = per_fold_summary["fold_roc_auc"].mean()
fold_std_roc_auc = per_fold_summary["fold_roc_auc"].std()
fold_mean_bal_acc = per_fold_summary["fold_balanced_accuracy"].mean()
fold_std_bal_acc = per_fold_summary["fold_balanced_accuracy"].std()
fold_mean_f1 = per_fold_summary["fold_f1"].mean()
fold_std_f1 = per_fold_summary["fold_f1"].std()
fold_mean_acc = per_fold_summary["fold_accuracy"].mean()
fold_std_acc = per_fold_summary["fold_accuracy"].std()

pooled_roc_auc = roc_auc_score(pooled_df["true_label"], pooled_df["bilateral_prob"])
pooled_bal_acc = balanced_accuracy_score(pooled_df["true_label"], pooled_df["pred_label_optimal"])
pooled_f1 = f1_score(pooled_df["true_label"], pooled_df["pred_label_optimal"])
pooled_acc = accuracy_score(pooled_df["true_label"], pooled_df["pred_label_optimal"])

print(f"\nFold-mean +/- std (5-fold CV, the standard reporting convention):")
print(f"  roc_auc            = {fold_mean_roc_auc:.4f} +/- {fold_std_roc_auc:.4f}")
print(f"  balanced_accuracy  = {fold_mean_bal_acc:.4f} +/- {fold_std_bal_acc:.4f}")
print(f"  f1                 = {fold_mean_f1:.4f} +/- {fold_std_f1:.4f}")
print(f"  accuracy           = {fold_mean_acc:.4f} +/- {fold_std_acc:.4f}")

print(f"\nPooled (all 167 subjects, one out-of-fold prediction each):")
print(f"  roc_auc            = {pooled_roc_auc:.4f}")
print(f"  balanced_accuracy  = {pooled_bal_acc:.4f}")
print(f"  f1                 = {pooled_f1:.4f}")
print(f"  accuracy           = {pooled_acc:.4f}")


=== SECTION 9: Test Evaluation ===

--- Fold 0 test evaluation ---
  Optimal threshold (Youden's J=0.7500, from val): 0.4028
  Test (n=34): roc_auc=0.9600, balanced_accuracy=0.8644, f1=0.8936, accuracy=0.8529

--- Fold 1 test evaluation ---
  Optimal threshold (Youden's J=1.0000, from val): 0.3064
  Test (n=34): roc_auc=0.8311, balanced_accuracy=0.5711, f1=0.8364, accuracy=0.7353

--- Fold 2 test evaluation ---
  Optimal threshold (Youden's J=0.9000, from val): 0.5395
  Test (n=33): roc_auc=0.8750, balanced_accuracy=0.7847, f1=0.8444, accuracy=0.7879

--- Fold 3 test evaluation ---
  Optimal threshold (Youden's J=0.8000, from val): 0.7169
  Test (n=33): roc_auc=0.8935, balanced_accuracy=0.8750, f1=0.8571, accuracy=0.8182

--- Fold 4 test evaluation ---
  Optimal threshold (Youden's J=1.0000, from val): 0.5287
  Test (n=33): roc_auc=0.8333, balanced_accuracy=0.7569, f1=0.7500, accuracy=0.6970

Pooled predictions: 167 subjects (expected 167)
PASS: pooled predictions cover exactly 167 uni

## Section 10 — Head-to-Head Comparison

This is the notebook's central result: the bilateral 8-node subject-level
graph's fold-mean±std metrics against the already-known reference numbers
below (stated as fixed values, not recomputed):

| Model | Metric | Value |
|---|---|---|
| DualBranchGAT primary | fold-mean roc_auc (Phase 13) | 0.882 ± 0.056 |
| graph_only_base | fold-mean roc_auc (Phase 17) | 0.8831 |
| TCI-only | pooled roc_auc (Phase 14) | 0.859 |
| Fusion_B | pooled roc_auc (Phase 16) | 0.889 |


In [10]:
print("=== SECTION 10: Head-to-Head Comparison ===")

comparison_table = pd.DataFrame([
    {"model": "DualBranchGAT primary (Phase 13, fold-mean)", "roc_auc": "0.882 +/- 0.056"},
    {"model": "graph_only_base (Phase 17, fold-mean)", "roc_auc": "0.8831"},
    {"model": "TCI-only (Phase 14, pooled)", "roc_auc": "0.859"},
    {"model": "Fusion_B (Phase 16, pooled)", "roc_auc": "0.889"},
    {"model": "Bilateral graph -- fold-mean (this notebook)", "roc_auc": f"{fold_mean_roc_auc:.4f} +/- {fold_std_roc_auc:.4f}"},
    {"model": "Bilateral graph -- pooled (this notebook)", "roc_auc": f"{pooled_roc_auc:.4f}"},
])

print(comparison_table.to_string(index=False))


=== SECTION 10: Head-to-Head Comparison ===
                                       model           roc_auc
 DualBranchGAT primary (Phase 13, fold-mean)   0.882 +/- 0.056
       graph_only_base (Phase 17, fold-mean)            0.8831
                 TCI-only (Phase 14, pooled)             0.859
                 Fusion_B (Phase 16, pooled)             0.889
Bilateral graph -- fold-mean (this notebook) 0.8786 +/- 0.0528
   Bilateral graph -- pooled (this notebook)            0.8596


## Section 11 — Save Results

Three output files, all under `results/` (never overwriting anything
under `data/processed/`):
- `bilateral_graph_results.csv` — fold-mean±std and pooled metrics.
- `bilateral_pooled_predictions.csv` — 167 rows: subject_id, group,
  true_label, bilateral_prob, fold.
- `bilateral_graph_summary.json` — the same numbers plus the fixed
  reference values from Section 10, machine-readable.


In [11]:
print("=== SECTION 11: Save Results ===")

results_summary_df = pd.DataFrame([{
    "roc_auc_fold_mean": fold_mean_roc_auc,
    "roc_auc_fold_std": fold_std_roc_auc,
    "balanced_accuracy_fold_mean": fold_mean_bal_acc,
    "balanced_accuracy_fold_std": fold_std_bal_acc,
    "f1_fold_mean": fold_mean_f1,
    "f1_fold_std": fold_std_f1,
    "accuracy_fold_mean": fold_mean_acc,
    "accuracy_fold_std": fold_std_acc,
    "roc_auc_pooled": pooled_roc_auc,
    "balanced_accuracy_pooled": pooled_bal_acc,
    "f1_pooled": pooled_f1,
    "accuracy_pooled": pooled_acc,
}])
results_path = RESULTS_DIR / "bilateral_graph_results.csv"
results_summary_df.to_csv(results_path, index=False)

pooled_predictions_df = pooled_df[["subject_id", "group", "true_label", "bilateral_prob", "fold"]].copy()
pooled_predictions_path = RESULTS_DIR / "bilateral_pooled_predictions.csv"
pooled_predictions_df.to_csv(pooled_predictions_path, index=False)

bilateral_summary = {
    "roc_auc_fold_mean": float(fold_mean_roc_auc),
    "roc_auc_fold_std": float(fold_std_roc_auc),
    "balanced_accuracy_fold_mean": float(fold_mean_bal_acc),
    "balanced_accuracy_fold_std": float(fold_std_bal_acc),
    "f1_fold_mean": float(fold_mean_f1),
    "f1_fold_std": float(fold_std_f1),
    "accuracy_fold_mean": float(fold_mean_acc),
    "accuracy_fold_std": float(fold_std_acc),
    "roc_auc_pooled": float(pooled_roc_auc),
    "balanced_accuracy_pooled": float(pooled_bal_acc),
    "f1_pooled": float(pooled_f1),
    "accuracy_pooled": float(pooled_acc),
    "extraction_path_used": extraction_path_used,
    "mc_dropout_used": False,
    "reference_values": {
        "DualBranchGAT_primary_phase13": {"roc_auc_fold_mean": 0.882, "roc_auc_fold_std": 0.056},
        "graph_only_base_phase17": {"roc_auc_fold_mean": 0.8831},
        "TCI_only_phase14": {"roc_auc_pooled": 0.859},
        "Fusion_B_phase16": {"roc_auc_pooled": 0.889},
    },
    "per_fold": [
        {"fold": fr["fold"], "best_epoch": fr["best_epoch"], "best_val_roc_auc": float(fr["best_val_roc_auc"])}
        for fr in fold_results
    ],
}

summary_path = RESULTS_DIR / "bilateral_graph_summary.json"
with open(summary_path, "w") as f:
    json.dump(bilateral_summary, f, indent=2)

print(f"Saved: {results_path}")
print(f"Saved: {pooled_predictions_path}")
print(f"Saved: {summary_path}")


=== SECTION 11: Save Results ===
Saved: results\bilateral_graph_results.csv
Saved: results\bilateral_pooled_predictions.csv
Saved: results\bilateral_graph_summary.json


## Key Findings (fill in after review)

- Does the bilateral graph beat the independent-per-foot `graph_only_base`
  (fold-mean roc_auc 0.8831)?
- Does it beat the full dual-branch primary model (fold-mean roc_auc
  0.882 ± 0.056)?
- Is any gap meaningful relative to the fold-to-fold std, per this
  project's established standard for judging that (Phase 17 treated gaps
  within roughly the ~0.05 AUC fold-to-fold noise floor as not clearly
  meaningful)?
- State the actual answer plainly either way, based on the printed
  Section 10 comparison table above.


## Section 13 — Final Summary


In [12]:
print("=== SECTION 13: Final Summary ===")

print(comparison_table.to_string(index=False))

print(f"\nExtraction path used in smoke test (Section 4): {extraction_path_used}")

n_checkpoints = len(list(CHECKPOINT_DIR.glob("bilateral_fold*_best.pt")))
print(f"\nCheckpoints saved: {n_checkpoints} / 5")

result_files = [
    RESULTS_DIR / "bilateral_graph_results.csv",
    RESULTS_DIR / "bilateral_pooled_predictions.csv",
    RESULTS_DIR / "bilateral_graph_summary.json",
]
for rf in result_files:
    print(f"  {rf}: {'saved' if rf.exists() else 'MISSING'}")

issues = []
if n_checkpoints != 5:
    issues.append(f"Only {n_checkpoints}/5 bilateral fold checkpoints found.")
for rf in result_files:
    if not rf.exists():
        issues.append(f"Missing result file: {rf}")

print()
if issues:
    print("ISSUES FOUND:")
    for i in issues:
        print(f"  - {i}")
else:
    print("0 issues found.")


=== SECTION 13: Final Summary ===
                                       model           roc_auc
 DualBranchGAT primary (Phase 13, fold-mean)   0.882 +/- 0.056
       graph_only_base (Phase 17, fold-mean)            0.8831
                 TCI-only (Phase 14, pooled)             0.859
                 Fusion_B (Phase 16, pooled)             0.889
Bilateral graph -- fold-mean (this notebook) 0.8786 +/- 0.0528
   Bilateral graph -- pooled (this notebook)            0.8596

Extraction path used in smoke test (Section 4): imported utility (src.model.sparse_to_dense_graph)

Checkpoints saved: 5 / 5
  results\bilateral_graph_results.csv: saved
  results\bilateral_pooled_predictions.csv: saved
  results\bilateral_graph_summary.json: saved

0 issues found.
